# Redes convolucionales para series de tiempo

En este ejercicio trabajamos con la serie del indice `S&P 500` para construir una red convolucional `CNN1D` aplicada a una serie de tiempo financiera.

El flujo replica el material base en cinco etapas:

1. lectura y preparacion de `SP500.csv`,
2. creacion de secuencias con `padding`,
3. escalado y definicion de la `CNN1D`,
4. optimizacion de hiperparametros con `RandomizedSearchCV`,
5. prediccion de retornos y reconstruccion aproximada del precio.


## Paso 1. Leer los datos del S&P 500


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

base_dir = Path.cwd().resolve()
if base_dir.name == 'notebooks':
    project_dir = base_dir.parent
else:
    project_dir = base_dir

sp500_path = project_dir / 'SP500.csv'
sp500_data = pd.read_csv(sp500_path, parse_dates=['Date'], index_col='Date')
print(sp500_data.head())


In [ ]:
# Calcular el rendimiento logaritmico
log_returns = np.log(sp500_data['price'] / sp500_data['price'].shift(1))

# Crear un DataFrame con los resultados
log_returns = log_returns.to_frame(name='log_yield')
log_returns['price'] = sp500_data['price']
log_returns = log_returns.dropna()

# Save the file
log_returns.to_csv(sp500_path)


In [ ]:
log_returns.describe()


In [ ]:
import plotly.express as px

fig = px.line(log_returns, x=log_returns.index, y='price', title='Precio del S&P 500')
fig.update_layout(xaxis_title='Fecha', yaxis_title='Precio')
fig.show()


In [ ]:
fig = px.line(log_returns, x=log_returns.index, y='log_yield', title='Retornos Logaritmicos del S&P 500')
fig.update_layout(xaxis_title='Fecha', yaxis_title='Log Yield')
fig.show()


In [ ]:
print(log_returns.head())


## Crear las secuencias

Las secuencias cuando no existe el numero total de datos para completar muestras de tamano exacto se deben complementar. Por eso se usa `padding`.


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


def create_sequences_with_padding(data, sequence_length):
    sequences = []
    targets = []
    padded_data = np.pad(data, (sequence_length - 1, 0), mode='constant', constant_values=0)
    for i in range(len(data)):
        seq = padded_data[i:i + sequence_length]
        target = data.iloc[i]
        sequences.append(seq)
        targets.append(target)
    return np.array(sequences), np.array(targets)


sequence_length = 14
train_sequences, train_targets = create_sequences_with_padding(log_returns['log_yield'], sequence_length)

print('Numero de secuencias:', len(train_sequences))
print('Primeras 3 secuencias:')
print(train_sequences[:3])


## Escalar datos


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_x = MinMaxScaler(feature_range=(-1, 1))
X_train_scaled = scaler_x.fit_transform(train_sequences.reshape(-1, train_sequences.shape[-1])).reshape(train_sequences.shape)

scaler_y = MinMaxScaler(feature_range=(-1, 1))
y_train_scaled = scaler_y.fit_transform(train_targets.reshape(-1, 1)).flatten()

print('X_train_scaled shape:', X_train_scaled.shape)
print('y_train_scaled shape:', y_train_scaled.shape)


## Crear el modelo

### Tasa de aprendizaje autoadaptativa

Ayuda al algoritmo `ADAM` a reducir la tasa cuando no hay mejoria en la perdida.


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
)


### Estructura minimo viable para una red CNN en series de tiempo


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.regularizers import l2

X_train_cnn = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))


def build_model(filters=32, kernel_size=3, dropout_rate=0.2, l2_reg=0.01):
    model = Sequential([
        Input(shape=(X_train_cnn.shape[1], 1)),
        Conv1D(filters=filters, kernel_size=kernel_size, activation='relu', kernel_regularizer=l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        Flatten(),
        Dense(1, activation='linear', kernel_regularizer=l2(l2_reg))
    ])
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model


### Uso de Random Search para optimizacion de hiperparametros


In [ ]:
param_distributions = {
    'model__filters': [16, 32, 64],
    'model__kernel_size': [2, 3, 4, 5],
    'model__dropout_rate': [0.1, 0.2, 0.3],
    'model__l2_reg': [0.001, 0.01, 0.1],
    'batch_size': [16, 32, 64],
    'epochs': [20, 30, 40]
}


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scikeras.wrappers import KerasRegressor

model_wrapper = KerasRegressor(
    model=build_model,
    verbose=0,
    fit__callbacks=[reduce_lr],
    fit__validation_split=0.2,
)

tscv = TimeSeriesSplit(n_splits=5)
print('Numero de divisiones (splits):', tscv.get_n_splits())

last_train_index = None
last_test_index = None
for train_index, test_index in tscv.split(X_train_cnn):
    X_train_fold, X_test_fold = X_train_cnn[train_index], X_train_cnn[test_index]
    y_train_fold, y_test_fold = y_train_scaled[train_index], y_train_scaled[test_index]
    print('Tamano del conjunto de entrenamiento:', X_train_fold.shape, y_train_fold.shape)
    last_train_index = train_index
    last_test_index = test_index


In [ ]:
import tensorflow as tf

# Fijar semillas para reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

random_search = RandomizedSearchCV(
    estimator=model_wrapper,
    param_distributions=param_distributions,
    n_iter=20,
    cv=tscv,
    verbose=0,
    n_jobs=1,
    random_state=42,
)

random_search.fit(X_train_cnn, y_train_scaled)
print('Mejores hiperparametros encontrados:', random_search.best_params_)


In [ ]:
best_model = random_search.best_estimator_.model_

history = best_model.fit(
    X_train_cnn,
    y_train_scaled,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[reduce_lr],
    verbose=0,
)

import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history['loss'], mode='lines', name='Training Loss'))
fig.add_trace(go.Scatter(y=history.history['val_loss'], mode='lines', name='Validation Loss'))
fig.update_layout(
    title='Training and Validation Loss over Epochs',
    xaxis_title='Epochs',
    yaxis_title='Loss'
)
fig.show()

print(f'Ultimo training loss: {history.history["loss"][-1]:.6f}')
print(f'Ultimo validation loss: {history.history["val_loss"][-1]:.6f}')


### Predicciones en el entrenamiento


In [ ]:
import matplotlib.pyplot as plt

train_predictions_scaled = best_model.predict(X_train_cnn, verbose=0)
train_predictions = scaler_y.inverse_transform(train_predictions_scaled.reshape(-1, 1)).flatten()
train_actual = scaler_y.inverse_transform(y_train_scaled.reshape(-1, 1)).flatten()

plt.figure(figsize=(12, 6))
plt.plot(train_actual, label='Valores reales', color='blue')
plt.plot(train_predictions, label='Predicciones', color='orange', linestyle='--')
plt.title('Predicciones vs Valores reales (Conjunto de entrenamiento)')
plt.xlabel('Indice')
plt.ylabel('Valor')
plt.legend()
plt.show()


### Predicciones en test


In [ ]:
test_predictions_scaled = best_model.predict(X_train_cnn[last_test_index], verbose=0)

test_predictions = scaler_y.inverse_transform(test_predictions_scaled.reshape(-1, 1)).flatten()
test_actual = scaler_y.inverse_transform(y_train_scaled[last_test_index].reshape(-1, 1)).flatten()

plt.figure(figsize=(12, 6))
plt.plot(test_actual, label='Valores reales', color='blue')
plt.plot(test_predictions, label='Predicciones', color='orange', linestyle='--')
plt.title('Predicciones vs Valores reales (Conjunto de prueba)')
plt.xlabel('Indice')
plt.ylabel('Valor')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(test_actual, test_predictions)
mse = mean_squared_error(test_actual, test_predictions)

print(f'Error absoluto medio (MAE): {mae}')
print(f'Error cuadratico medio (MSE): {mse}')


### Prediccion en el problema completo

Prediccion del valor del indice `SP500` usando la rentabilidad pronosticada y la media movil del precio en el conjunto de test.


In [ ]:
initial_price = log_returns['price'].iloc[last_train_index[-1]]
test_prices = [initial_price]

for ret in test_predictions:
    next_price = test_prices[-1] * (1 + ret)
    test_prices.append(next_price)

test_prices = np.array(test_prices[1:])
window_size = 5
smoothed_prices = np.convolve(test_prices, np.ones(window_size) / window_size, mode='valid')

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(test_prices))),
    y=test_prices,
    mode='lines',
    name='Precios pronosticados',
    line=dict(color='blue', width=2)
))
fig.add_trace(go.Scatter(
    x=list(range(window_size - 1, len(test_prices))),
    y=smoothed_prices,
    mode='lines',
    name='Precios suavizados (Media movil)',
    line=dict(color='orange', width=2, dash='dash')
))
fig.update_layout(
    title='Precios pronosticados vs Precios suavizados',
    xaxis_title='Indice',
    yaxis_title='Precio',
    hovermode='x unified',
    template='plotly_white',
    width=1200,
    height=600,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()


## Cierre

Este inciso reproduce una estrategia `CNN1D` para detectar patrones locales de corto plazo sobre rendimientos logaritmicos del `S&P 500`. La busqueda aleatoria de hiperparametros y la tasa de aprendizaje autoadaptativa permiten construir una arquitectura razonable sin fijar a mano todos los componentes del modelo.
